In [ ]:
import scanpy as sc
import scvi
import sys
sys.path.append("/multiHIVE/src")
from multiHIVE import multiHIVE
import torch

In [3]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
torch.set_float32_matmul_precision("high")

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [ ]:
adata = sc.read_h5ad( "/Data/Brain-ISSAAC/Brain-ISSAAC-seq.h5ad")
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 10361 × 197472
    obs: 'cell_type', 'batch'
    var: 'modality'

In [5]:
del adata.obsm
del adata.obsp

In [ ]:
adata = scvi.data.organize_multiome_anndatas(adata)
adata = adata[:, adata.var["modality"].argsort()].copy()
sc.pp.filter_genes(adata, min_cells=int(adata.shape[0] * 0.01))
multiHIVE.setup_anndata(adata, batch_key="modality")
adata

In [ ]:
vae = multiHIVE(adata, latent_distribution="normal",
                n_genes=(adata.var["modality"] == "Gene Expression").sum(),
                n_regions=(adata.var["modality"] == "Peaks").sum(),
                n_proteins=0,
                kl_dot_product  = True,
                deep_network = False,
               )

In [ ]:
vae.train(max_epochs=500, early_stopping=False)

In [9]:
vae.get_latent_representation()

In [11]:
adata.write("./outputs/Brain-ISSAAC/multiHIVE.h5ad")